In [3]:
import pandas as pd
from IPython.core.magics import display

df = pd.read_csv('../data/interim/tracklist.csv')

In [4]:
print(f"Total rows       : {len(df):,}")
print(f"Unique DJs       : {df['dj_name'].nunique()}")
print(f"Unique mixes     : {df['mix_id'].nunique()}")
print(f"Unique tracks    : {df['track_id'].nunique()}")
print(f"Genres           : {sorted(df['genre'].dropna().unique())}")
print(f"Play types       : {df['play_type'].value_counts().to_dict()}")

Total rows       : 130,167
Unique DJs       : 49
Unique mixes     : 5870
Unique tracks    : 58244
Genres           : ['afro house', 'drum and base', 'melodic house', 'tech house', 'techno', 'trance']
Play types       : {'sequential': 125552, 'simultaneous': 4615}


In [5]:
mixes_per_dj = (
    df.groupby('dj_name')['mix_id'].nunique()
    .sort_values(ascending=False)
    .rename('n_mixes')
)
print(mixes_per_dj.to_string())
print(f"\nAvg mixes/DJ: {mixes_per_dj.mean():.1f}  |  Max: {mixes_per_dj.max()}  |  Min: {mixes_per_dj.min()}")

dj_name
Adambeyer            250
Arminvanbuuren       250
Alyandfila           250
Sultanplusshepard    250
Noisia               250
Johnocallaghan       250
Johnsummit           250
Markusschulz         250
Ferrycorsten         250
Paulvandyk           250
Carlcox              248
Solomun              241
Charlottedewitte     223
Chrislake            213
Amelielens           199
Hotsince82           179
Fisher               159
Taleofus             155
Rank1                140
Shimza               136
Subfocus             132
Moblack              128
Anyma                107
Rufusdusol           102
Black Coffee          88
Pendulum              84
Chasestatus           79
Dimension             78
Massano               76
Argy                  63
Djtennis              63
Saralandry            62
Benbohmer             62
Andyc                 52
Themba                37
Monolink              33
Klangkuenstler        29
Hedex                 24
Ltjbukem              23
Enoo Napa        

In [6]:
tracks_per_dj = (
    df.groupby('dj_name')['track_id'].count()
    .sort_values(ascending=False)
    .rename('n_track_rows')
)
print(tracks_per_dj.to_string())
print(f"\nAvg tracks/DJ: {tracks_per_dj.mean():.0f}  |  Max: {tracks_per_dj.max()}  |  Min: {tracks_per_dj.min()}")

dj_name
Arminvanbuuren       7478
Markusschulz         7243
Johnsummit           6650
Noisia               6443
Paulvandyk           6331
Alyandfila           6249
Adambeyer            5765
Carlcox              5282
Subfocus             5066
Chrislake            4838
Johnocallaghan       4688
Ferrycorsten         4598
Solomun              4583
Fisher               4031
Charlottedewitte     3656
Hotsince82           3592
Amelielens           3582
Sultanplusshepard    3107
Andyc                3053
Dimension            2966
Taleofus             2548
Anyma                2536
Chasestatus          2436
Rank1                2176
Pendulum             2071
Saralandry           1946
Shimza               1785
Rufusdusol           1663
Massano              1464
Black Coffee         1370
Argy                 1315
Hedex                1188
Moblack              1130
Djtennis             1091
Benbohmer             934
Ihatemodels           697
Themba                565
Calibre               534
Holy

In [8]:
seq = df[df['play_type'] == 'sequential']
tracks_per_mix = seq.groupby('mix_id')['track_id'].count().sort_values(ascending=False).rename('n_sequential_tracks')
print(tracks_per_mix.describe().round(1))
print(f"\nTotal sequential track rows: {len(seq):,}")

count    5870.0
mean       21.3
std        15.1
min         0.0
25%        12.0
50%        19.0
75%        27.0
max       406.0
Name: n_sequential_tracks, dtype: float64

Total sequential track rows: 125,552


In [11]:
genre_stats = (
    df.groupby('genre')
    .agg(
        n_djs   = ('dj_name', 'nunique'),
        n_mixes = ('mix_id', 'nunique'),
        n_tracks= ('track_id', 'count'),
    )
    .sort_values('n_tracks', ascending=False)
)
print(genre_stats)


               n_djs  n_mixes  n_tracks
genre                                  
trance             7     1640     38763
tech house         8     1368     30221
drum and base      9      741     24218
techno             8      814     16926
melodic house      8      848     13927
afro house         9      459      5870


In [15]:
sim = df[df['play_type'] == 'simultaneous']
seq = df[df['play_type'] == 'sequential']
mix_info = df[['mix_id', 'dj_name', 'genre']].drop_duplicates('mix_id')

print(f"Sequential rows  : {len(seq):,}")
print(f"Simultaneous rows: {len(sim):,}  ({len(sim)/len(df)*100:.1f}% of all rows)")
print(f"Mixes with simultaneous tracks: {sim['mix_id'].nunique()} / {df['mix_id'].nunique()}\n")

# --- simultaneous count per mix, joined with dj/genre ---
sim_per_mix = (
    sim.groupby('mix_id').size()
    .reset_index(name='n_simultaneous')
    .merge(mix_info, on='mix_id')
)

# by genre
print("=== By genre ===")
print(
    sim_per_mix.groupby('genre')['n_simultaneous']
    .agg(n_mixes_affected='count', total_sim_tracks='sum', avg_per_mix='mean')
    .round(1).sort_values('total_sim_tracks', ascending=False)
)

# by DJ
print("=== By DJ ===")
print(
    sim_per_mix.groupby(['dj_name', 'genre'])['n_simultaneous']
    .agg(n_mixes_with_sim='count', total_sim_tracks='sum')
    .sort_values('total_sim_tracks', ascending=False)
    .reset_index()
)

# mixes ranked by simultaneous density (% of all tracks in mix that are overlays)
mix_seq_counts = seq.groupby('mix_id').size().reset_index(name='n_sequential')
print("=== Top 20 mixes by simultaneous density ===")
print(
    sim_per_mix
    .merge(mix_seq_counts, on='mix_id', how='left')
    .assign(sim_pct=lambda x: (x['n_simultaneous'] / (x['n_sequential'] + x['n_simultaneous']) * 100).round(1))
    .sort_values('sim_pct', ascending=False)
    [['dj_name', 'genre', 'mix_id', 'n_sequential', 'n_simultaneous', 'sim_pct']]
    .head(20).reset_index(drop=True)
)

# overlay pairs — what track was played on top of what
parent_lookup = (
    seq[['mix_id', 'track_id', 'track_name', 'artist_name']]
    .rename(columns={'track_id': 'overlay_parent', 'track_name': 'parent_track', 'artist_name': 'parent_artist'})
)
overlay_pairs = (
    sim[sim['overlay_parent'].notna()]
    [['mix_id', 'track_id', 'track_name', 'artist_name', 'overlay_parent']]
    .rename(columns={'track_id': 'overlay_track_id', 'track_name': 'overlay_track', 'artist_name': 'overlay_artist'})
    .merge(parent_lookup, on=['mix_id', 'overlay_parent'], how='left')
    .merge(mix_info, on='mix_id')
)
print(f"=== Overlay pairs resolved: {len(overlay_pairs):,} ===")
print(overlay_pairs[['genre', 'dj_name', 'parent_track', 'parent_artist', 'overlay_track', 'overlay_artist']].head(20))

Sequential rows  : 125,552
Simultaneous rows: 4,615  (3.5% of all rows)
Mixes with simultaneous tracks: 1195 / 5870

=== By genre ===
               n_mixes_affected  total_sim_tracks  avg_per_mix
genre                                                         
drum and base               287              2527          8.8
tech house                  315               655          2.1
trance                      198               541          2.7
melodic house               162               309          1.9
techno                      143               306          2.1
afro house                   90               277          3.1
=== By DJ ===
              dj_name          genre  n_mixes_with_sim  total_sim_tracks
0               Andyc  drum and base                32               718
1            Subfocus  drum and base                79               612
2           Dimension  drum and base                54               443
3               Hedex  drum and base                17  

In [18]:
seq = df[df['play_type'] == 'sequential']

unique_seq_tracks = seq['track_id'].nunique()
unique_sim_tracks = df[df['play_type'] == 'simultaneous']['track_id'].nunique()
overlap = len(set(seq['track_id']) & set(df[df['play_type'] == 'simultaneous']['track_id']))

print(f"Sequential track rows       : {len(seq):,}")
print(f"Unique sequential track_ids : {unique_seq_tracks:,}  ← available for MERT + contrastive training")
print(f"Unique simultaneous track_ids: {unique_sim_tracks:,}")
print(f"  └─ also appear in sequential : {overlap:,}  ({overlap/unique_sim_tracks*100:.0f}% already covered)")
print(f"  └─ sim-only (never sequential): {unique_sim_tracks - overlap:,}")

print("\n=== Unique sequential tracks per genre ===")
print(
    seq.groupby('genre')['track_id']
    .nunique()
    .rename('unique_tracks')
    .sort_values(ascending=False)
    .to_frame()
    .assign(pct=lambda x: (x['unique_tracks'] / x['unique_tracks'].sum() * 100).round(1))
)


Sequential track rows       : 125,552
Unique sequential track_ids : 57,307  ← available for MERT + contrastive training
Unique simultaneous track_ids: 2,152
  └─ also appear in sequential : 1,215  (56% already covered)
  └─ sim-only (never sequential): 937

=== Unique sequential tracks per genre ===
               unique_tracks   pct
genre                             
trance                 16125  26.3
tech house             14996  24.5
drum and base          11962  19.5
techno                  8417  13.7
melodic house           6034   9.9
afro house              3708   6.1
